### Run multi-thermal ALMANAC for a number of detection events and create log files.

In [1]:
import subprocess, json, os
from datetime import datetime
import time

os.makedirs("logs", exist_ok=True)

total_start = time.perf_counter()

def log(msg):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    with open("logs/main.log", "a") as f:
        f.write(f"[{timestamp}] {msg}\n")
    print(msg)  # also print to console

def extract_json(text: str):
    start = text.find('[')
    end = text.rfind(']') + 1
    if start == -1 or end == -1:
        raise RuntimeError(f"Could not find JSON in output:\n{text}")
    return json.loads(text[start:end])

def run_single_cme(cme_time):
    result = subprocess.run(
        ["python3", "run_single_cme_worker.py", cme_time],
        capture_output=True,
        text=True
    )

    # save worker stdout/stderr to a separate log
    worker_log_file = f"logs/worker_subprocess_{cme_time.replace(':','').replace(' ','_')}.log"
    with open(worker_log_file, "w") as f:
        f.write("STDOUT:\n" + result.stdout + "\n")
        f.write("STDERR:\n" + result.stderr + "\n")

    out = result.stdout.strip()

    if not out:
        raise RuntimeError(f"Worker produced no output.\nSee {worker_log_file}")

    data = extract_json(out)

    # check if worker returned an error object
    if isinstance(data[0], dict) and "error" in data[0]:
        raise RuntimeError(
            f"Worker reported an exception for CME {cme_time}:\n"
            f"{data[0]['type']}: {data[0]['error']}\n"
            f"See logs/worker_{cme_time.replace(':','').replace(' ','_')}.log for details."
        )

    return data  # detect, assoc_json


# ==========================
# Main CME loop
# ==========================

cmes = [
        '2010-08-14 10:12',
        '2011-02-15 02:24',
        '2011-09-06 02:24',
        '2011-11-26 07:12',
        '2012-04-07 21:15',
        '2012-11-23 13:48',
        '2012-11-27 02:36',
        '2013-03-15 07:12',
        '2013-04-11 07:24',
        '2013-05-15 07:12',
        '2013-07-09 15:12',
        '2013-08-20 08:12',
        '2013-09-29 22:12',
        '2014-02-18 01:36',
        '2014-03-23 03:36',
        '2014-04-01 16:48',
        '2014-04-29 23:24',
        '2014-08-15 17:48',
        '2014-08-22 11:12',
        '2014-12-21 12:12'
]

detections = []
assoc_files = []

for cme in cmes:
    log(f"Running {cme}")
    try:
        detect, assoc = run_single_cme(cme)
        detections.append(detect)
        assoc_files.append(assoc)
        log(f"Completed {cme}\n{'='*75}")
    except Exception as e:
        log(f"FAILED {cme}: {e}\n{'='*75}")


total_end = time.perf_counter()

print(f'Total Processing time for {len(cmes)} regions: {total_end - total_start:.2f} seconds')

Running 2010-08-14 10:12
Completed 2010-08-14 10:12
Running 2011-02-15 02:24
Completed 2011-02-15 02:24
Running 2011-09-06 02:24
Completed 2011-09-06 02:24
Running 2011-11-26 07:12
Completed 2011-11-26 07:12
Running 2012-04-07 21:15
Completed 2012-04-07 21:15
Running 2012-11-23 13:48
Completed 2012-11-23 13:48
Running 2012-11-27 02:36
Completed 2012-11-27 02:36
Running 2013-03-15 07:12
Completed 2013-03-15 07:12
Running 2013-04-11 07:24
Completed 2013-04-11 07:24
Running 2013-05-15 07:12
Completed 2013-05-15 07:12
Running 2013-07-09 15:12
Completed 2013-07-09 15:12
Running 2013-08-20 08:12
Completed 2013-08-20 08:12
Running 2013-09-29 22:12
Completed 2013-09-29 22:12
Running 2014-02-18 01:36
Completed 2014-02-18 01:36
Running 2014-03-23 03:36
Completed 2014-03-23 03:36
Running 2014-04-01 16:48
Completed 2014-04-01 16:48
Running 2014-04-29 23:24
Completed 2014-04-29 23:24
Running 2014-08-15 17:48
Completed 2014-08-15 17:48
Running 2014-08-22 11:12
Completed 2014-08-22 11:12
Running 2014

### Run single instance of multi-thermal ALMANAC where logs are printed

In [1]:
lasco_time = ['2025-11-11 10:30']

detections, assoc_files = [], []
from run_almanac import run_almanac
detect, assoc = run_almanac(
    end_time=lasco_time,    # CME-window end time
    cadence=6,              # minutes
    tRange=6,               # hours
    minCluster=4            # minimum single-channel detections
)

Reading FITS: 100%|██████████| 61/61 [00:02<00:00, 22.90 file/s]


Channel 335 completed
Channel 94 completed
Channel 193 completed
Channel 131 completed
Channel 304 completed
Channel 171 completed
Channel 211 completed

Deleted: almanac_out/2025/11/almanac_20251111_0430_wvlnth_171_1.pkl.gz not in any cluster > 4 files
Deleted: almanac_out/2025/11/almanac_20251111_0430_wvlnth_171_3.pkl.gz not in any cluster > 4 files
Deleted: almanac_out/2025/11/almanac_20251111_0430_wvlnth_193_6.pkl.gz not in any cluster > 4 files
Deleted: almanac_out/2025/11/almanac_20251111_0454_wvlnth_171_4.pkl.gz not in any cluster > 4 files
Deleted: almanac_out/2025/11/almanac_20251111_0747_wvlnth_94_1.pkl.gz not in any cluster > 4 files
Deleted: almanac_out/2025/11/almanac_20251111_0748_wvlnth_131_1.pkl.gz not in any cluster > 4 files
Deleted: almanac_out/2025/11/almanac_20251111_0918_wvlnth_304_2.pkl.gz not in any cluster > 4 files
Deleted: almanac_out/2025/11/almanac_20251111_0947_wvlnth_211_1.pkl.gz not in any cluster > 4 files
Deleted: almanac_out/2025/11/almanac_20251111_1